# gen-gec-errant Quickstart

This notebook shows two approaches:
1. **Per-stage**: construct configs in Python, call runners individually
2. **Full pipeline**: load a YAML config and call `run_pipeline`

## Approach 1: Per-stage

In [ ]:
from gen_gec_errant.data_loader import run_data_loader
from gen_gec_errant.data_loader.config import DataLoaderConfig

# Configure and run data loading
dl_config = DataLoaderConfig(
    data_path="../data/efcamdat_sentences.txt",
    max_sentences=20,
    min_words=10,
)
items = run_data_loader(dl_config)
print(f"Loaded {len(items)} items")
items[0]

In [ ]:
from gen_gec_errant.generation import run_generation
from gen_gec_errant.generation.config import GenerationConfig, GenerationParams, ModelConfig

# Configure and run generation
gen_config = GenerationConfig(
    params=GenerationParams(max_new_tokens=30, temperature=1.0),
    model=ModelConfig(name="gpt2-base", hf_model_id="gpt2"),
    batch_size=4,
    device="auto",
)
results = run_generation(gen_config, items)
print(f"Generated {len(results['continuations'])} continuations")

In [ ]:
from gen_gec_errant.gec import run_gec
from gen_gec_errant.gec.config import GECConfig

# Run GEC
gec_config = GECConfig(method="dedicated", device="auto")
results = run_gec(gec_config, results)
print(f"Corrected {len(results['corrected_continuations'])} sentences")

In [ ]:
from gen_gec_errant.annotation import run_annotation
from gen_gec_errant.annotation.config import AnnotationConfig

# Run ERRANT annotation
ann_config = AnnotationConfig(lang="en")
results = run_annotation(ann_config, results)
print(f"Error summary: {results['error_summary']}")

## Approach 2: Full pipeline from YAML

In [ ]:
from gen_gec_errant.pipeline import run_pipeline
from gen_gec_errant.pipeline.config import load_config_from_yaml

config = load_config_from_yaml("../configs/pipeline/quickstart.yaml")
summaries, comparison = run_pipeline(config)

for s in summaries:
    print(f"{s['model_name']}: PPL={s['ppl_mean']:.2f}, Errors={s['avg_errors_per_sentence']:.2f}")